<a href="https://colab.research.google.com/github/andrearomano-collab/ML_oxidation_notebooks/blob/main/ML_oxidation_grouped_profiles_38_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anchor-grouped time profiles of 38 selected mass-spectral features

This standalone notebook reproduces four grouped time-profile graphs from the extracted DHS and DSI measurement CSV files, their `log.csv` files and the target-to-anchor mapping in `55_most_aboundant_features_annotated_final.xlsx`.

It contains only the operations required for these graphs:

1. read and validate the 38 target-variable mappings;
2. calculate time-window-averaged, reagent-ion-normalised signals;
3. subtract the matched normalised blank and replace negative values with zero;
4. average replicates at each oxidation time;
5. rescale each mean profile to the interval 0–1 using min–max scaling;
6. group targets by their corresponding anchor variable; and
7. create four graphs, using a thicker line for the anchor profile and thinner lines for correlated profiles.

No PCA, clustering, correlation testing, feature selection or unrelated exploratory plots are included. The target list is treated as an input because the 38 features and their anchor assignments have already been selected and annotated for the manuscript.


## Input files and configuration

Expected directory structure:

```text
PROJECT_ROOT/
├── ML_oxidation_grouped_profiles_38_features.ipynb
├── 55_most_aboundant_features_annotated_final.xlsx
└── source_data/
    ├── DHS/
    │   ├── log.csv
    │   └── extracted DHS CSV files
    └── DSI/
        ├── log.csv
        └── extracted DSI CSV files
```

Only measurement files named in the `File_Name` column of each log are read. In Google Colab, mount Google Drive if required and change `PROJECT_ROOT` to the directory containing these inputs.


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Optional Google Colab setup:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = Path('/content/drive/MyDrive/path/to/supplementary_package')

PROJECT_ROOT = Path('.')
DATA_ROOT = PROJECT_ROOT / 'source_data'
TARGET_LIST_PATH = PROJECT_ROOT / '55_most_aboundant_features_annotated_final.xlsx'
OUTPUT_ROOT = PROJECT_ROOT / 'grouped_profile_figures'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ANCHOR_ORDER = [
    'MZ_344.278442_DSI',
    'MZ_328.283234_DSI',
    'MZ_204.158478_DSI',
    'MZ_130.122467_DHS',
]

ANCHOR_LINE_WIDTH = 4.0
CORRELATED_LINE_WIDTH = 1.2

required_paths = [
    TARGET_LIST_PATH,
    DATA_ROOT / 'DHS' / 'log.csv',
    DATA_ROOT / 'DSI' / 'log.csv',
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f'Required input not found: {required_path}. Update PROJECT_ROOT before continuing.'
        )

print(f'Project root: {PROJECT_ROOT.resolve()}')
print(f'Target list: {TARGET_LIST_PATH.resolve()}')
print(f'Figure output directory: {OUTPUT_ROOT.resolve()}')


## Read and validate the target-to-anchor mapping

The spreadsheet is the authoritative list of plotted variables. The notebook verifies that it contains 38 unique targets assigned to the four expected anchors, that every target follows the `MZ_<mass>_<mode>` naming convention, and that every anchor occurs once as its own target.


In [ ]:
mapping_columns = ['Reference Variable', 'Target Variable']
target_mapping = pd.read_excel(
    TARGET_LIST_PATH,
    sheet_name='Sheet1',
    usecols=mapping_columns,
)
target_mapping = target_mapping.dropna(subset=mapping_columns).copy()
for column in mapping_columns:
    target_mapping[column] = target_mapping[column].astype(str).str.strip()

variable_pattern = re.compile(r'^MZ_([0-9]+(?:\.[0-9]+)?)_(DHS|DSI)$')


def parse_variable_name(variable_name):
    match = variable_pattern.fullmatch(variable_name)
    if match is None:
        raise ValueError(
            f"Invalid variable name '{variable_name}'. Expected MZ_<mass>_<DHS or DSI>."
        )
    return float(match.group(1)), match.group(2)


parsed_targets = target_mapping['Target Variable'].map(parse_variable_name)
target_mapping['Target m/z'] = parsed_targets.map(lambda value: value[0])
target_mapping['Mode'] = parsed_targets.map(lambda value: value[1])

if len(target_mapping) != 38:
    raise ValueError(f'Expected 38 mapping rows, obtained {len(target_mapping)}.')
if target_mapping['Target Variable'].nunique() != 38:
    raise ValueError('Target variables must be unique.')
if set(target_mapping['Reference Variable']) != set(ANCHOR_ORDER):
    raise ValueError('The mapping does not contain the four expected anchor variables.')

for anchor in ANCHOR_ORDER:
    self_rows = target_mapping[
        (target_mapping['Reference Variable'] == anchor)
        & (target_mapping['Target Variable'] == anchor)
    ]
    if len(self_rows) != 1:
        raise ValueError(f"Anchor '{anchor}' must occur exactly once as its own target.")

group_counts = (
    target_mapping.groupby('Reference Variable', sort=False)['Target Variable']
    .count()
    .reindex(ANCHOR_ORDER)
)
print('Validated target-to-anchor mapping:')
print(group_counts.rename('Number of plotted profiles').to_string())


## Functions for signal extraction, normalisation and blank subtraction

The calculations reproduce the preprocessing used in the original notebook.

- **DHS:** the end of the logged blank is the reference point (`rp`); `rp − 30` to `rp` is averaged for the blank and `rp + 1` to `rp + 60` for the sample.
- **DSI:** the logged `start` and `end` values are used directly.
- **Reagent-ion normalisation:** for every analytical file, the most intense feature in *m/z* 75.5–76.5 and the most intense feature in *m/z* 133.5–134.5 are identified from their whole-file means. Target-window means are divided by their summed mean signal and multiplied by 10⁶.
- **Background subtraction:** the matched normalised blank is subtracted from each sample. Negative results are set to zero.


In [ ]:
def extract_mz_from_col_name(column_name):
    """Extract the numerical m/z value from a Tofware column name."""
    marker = 'm/Q '
    marker_position = str(column_name).find(marker)
    if marker_position == -1:
        return None
    try:
        return float(str(column_name)[marker_position + len(marker):].strip("' "))
    except ValueError:
        return None


def find_feature_column(dataframe, target_mz, tolerance=1e-6):
    """Find the mass-feature column matching a specified exact m/z."""
    candidates = []
    for column in dataframe.columns:
        mz_value = extract_mz_from_col_name(column)
        if mz_value is not None:
            candidates.append((abs(mz_value - target_mz), column, mz_value))

    if not candidates:
        raise ValueError('No Tofware mass-feature columns were found.')

    difference, column, observed_mz = min(candidates, key=lambda item: item[0])
    if difference > tolerance:
        raise KeyError(
            f'No feature found for m/z {target_mz:.6f}; nearest was {observed_mz:.6f}.'
        )
    return column


def most_intense_feature_in_range(dataframe, lower_mz, upper_mz):
    """Return the feature with the largest whole-file mean in an m/z interval."""
    candidates = [
        column
        for column in dataframe.columns
        if (mz_value := extract_mz_from_col_name(column)) is not None
        and lower_mz <= mz_value <= upper_mz
    ]
    if not candidates:
        raise KeyError(f'No mass feature found between m/z {lower_mz} and {upper_mz}.')

    means = dataframe[candidates].mean(axis=0)
    selected_column = means.idxmax()
    return selected_column, float(means[selected_column])


def build_normalised_target_table(data_path, mode, targets):
    """Calculate time-window means and reagent-ion-normalised target signals."""
    log = pd.read_csv(data_path / 'log.csv')
    required_log_columns = {
        'sample', 'File_Name', 'hour', 'replicate', 'start', 'end'
    }
    missing_log_columns = required_log_columns.difference(log.columns)
    if missing_log_columns:
        raise KeyError(f'Missing log columns: {sorted(missing_log_columns)}')

    file_cache = {}
    file_information = {}

    for file_name in log['File_Name'].drop_duplicates():
        file_path = data_path / file_name
        if not file_path.exists():
            raise FileNotFoundError(f'Log-referenced measurement file not found: {file_path}')

        dataframe = pd.read_csv(file_path)
        if 't_elapsed_Buf' not in dataframe.columns:
            raise KeyError(f"Column 't_elapsed_Buf' not found in {file_path}")

        target_columns = {
            variable_name: find_feature_column(dataframe, target_mz)
            for variable_name, target_mz in targets
        }
        mz76_column, mz76_mean = most_intense_feature_in_range(dataframe, 75.5, 76.5)
        mz134_column, mz134_mean = most_intense_feature_in_range(dataframe, 133.5, 134.5)
        reagent_ion_signal = mz76_mean + mz134_mean
        if not np.isfinite(reagent_ion_signal) or reagent_ion_signal <= 0:
            raise ValueError(f'Invalid reagent-ion signal in {file_path}')

        file_cache[file_name] = dataframe
        file_information[file_name] = {
            'target_columns': target_columns,
            'reagent_columns': (mz76_column, mz134_column),
            'reagent_ion_signal': reagent_ion_signal,
        }

    blank_end_by_file = {}
    if mode == 'DHS':
        for file_name in log['File_Name'].drop_duplicates():
            blank_rows = log[
                (log['File_Name'] == file_name)
                & log['sample'].astype(str).str.endswith('_B')
            ]
            if blank_rows.empty:
                raise ValueError(f'No DHS blank entry found for {file_name}')
            blank_end_by_file[file_name] = float(blank_rows.iloc[0]['end'])

    rows = []
    for _, log_row in log.iterrows():
        file_name = log_row['File_Name']
        dataframe = file_cache[file_name]
        information = file_information[file_name]
        is_blank = str(log_row['sample']).endswith('_B')

        if mode == 'DHS':
            reference_point = blank_end_by_file[file_name]
            if is_blank:
                start_time, end_time = reference_point - 30, reference_point
            else:
                start_time, end_time = reference_point + 1, reference_point + 60
        elif mode == 'DSI':
            start_time = float(log_row['start'])
            end_time = float(log_row['end'])
        else:
            raise ValueError("mode must be either 'DHS' or 'DSI'")

        period = dataframe[
            dataframe['t_elapsed_Buf'].between(start_time, end_time, inclusive='both')
        ]
        if period.empty:
            raise ValueError(
                f'Empty {mode} averaging window for {file_name}: {start_time}–{end_time}'
            )

        output_row = {
            'File_Name': file_name,
            'Type': 'Blank' if is_blank else 'Sample',
            'Hour': int(log_row['hour']),
            'Replicate': int(log_row['replicate']),
        }
        for variable_name, target_column in information['target_columns'].items():
            mean_signal = float(period[target_column].mean())
            output_row[variable_name] = (
                mean_signal / information['reagent_ion_signal'] * 1e6
            )
        rows.append(output_row)

    table = pd.DataFrame(rows).sort_values(
        ['Hour', 'Replicate', 'File_Name', 'Type']
    ).reset_index(drop=True)
    return log, table, file_information


def subtract_matched_blanks(normalised_table, feature_columns):
    """Subtract matched normalised blanks and clip negative values to zero."""
    samples = normalised_table[normalised_table['Type'] == 'Sample'].copy()
    blanks = normalised_table[normalised_table['Type'] == 'Blank'].copy()

    paired = samples.merge(
        blanks,
        on=['File_Name', 'Hour', 'Replicate'],
        suffixes=('_sample', '_blank'),
        validate='one_to_one',
    )

    result = paired[['File_Name', 'Hour', 'Replicate']].copy()
    for feature_column in feature_columns:
        result[feature_column] = np.clip(
            paired[f'{feature_column}_sample'] - paired[f'{feature_column}_blank'],
            a_min=0,
            a_max=None,
        )

    return result.sort_values(['Hour', 'Replicate']).reset_index(drop=True)


## Calculate the 38 normalised, background-subtracted profiles

The mapping file determines which feature columns are extracted from DHS and DSI. No other mass-spectral variables are carried through the analysis. The assertions check the log-referenced file counts and the expected number of matched sample/blank observations.


In [ ]:
dhs_target_table = target_mapping[target_mapping['Mode'] == 'DHS']
dsi_target_table = target_mapping[target_mapping['Mode'] == 'DSI']

dhs_targets = list(zip(
    dhs_target_table['Target Variable'], dhs_target_table['Target m/z']
))
dsi_targets = list(zip(
    dsi_target_table['Target Variable'], dsi_target_table['Target m/z']
))

dhs_log, dhs_normalised, dhs_file_information = build_normalised_target_table(
    DATA_ROOT / 'DHS', 'DHS', dhs_targets
)
dsi_log, dsi_normalised, dsi_file_information = build_normalised_target_table(
    DATA_ROOT / 'DSI', 'DSI', dsi_targets
)

dhs_profiles = subtract_matched_blanks(
    dhs_normalised, dhs_target_table['Target Variable'].tolist()
)
dsi_profiles = subtract_matched_blanks(
    dsi_normalised, dsi_target_table['Target Variable'].tolist()
)

assert len(dhs_file_information) == 35
assert len(dsi_file_information) == 24
assert len(dhs_profiles) == 35
assert len(dsi_profiles) == 24
assert dhs_profiles['Hour'].nunique() == 12
assert dsi_profiles['Hour'].nunique() == 12

print(f'DHS: {len(dhs_target_table)} targets, {len(dhs_profiles)} replicate observations, '
      f'{dhs_profiles["Hour"].nunique()} oxidation times')
print(f'DSI: {len(dsi_target_table)} targets, {len(dsi_profiles)} replicate observations, '
      f'{dsi_profiles["Hour"].nunique()} oxidation times')


## Average replicates and rescale every profile to 0–1

Replicate-level DHS and DSI tables are converted to long format and combined. Replicates are then averaged at each oxidation time. Finally, each 12-point mean profile is independently transformed using

\[
x_{scaled}=\frac{x-x_{min}}{x_{max}-x_{min}}.
\]

Thus, each plotted profile has a minimum of 0 and a maximum of 1, while its temporal shape is retained.


In [ ]:
dhs_long = dhs_profiles.melt(
    id_vars=['Hour', 'Replicate'],
    value_vars=dhs_target_table['Target Variable'].tolist(),
    var_name='Target Variable',
    value_name='Background-subtracted intensity (ncps)',
)
dsi_long = dsi_profiles.melt(
    id_vars=['Hour', 'Replicate'],
    value_vars=dsi_target_table['Target Variable'].tolist(),
    var_name='Target Variable',
    value_name='Background-subtracted intensity (ncps)',
)

replicate_profiles = pd.concat([dhs_long, dsi_long], ignore_index=True)
replicate_profiles = replicate_profiles.merge(
    target_mapping[['Reference Variable', 'Target Variable']],
    on='Target Variable',
    how='left',
    validate='many_to_one',
)
if replicate_profiles['Reference Variable'].isna().any():
    raise ValueError('At least one reconstructed target lacks an anchor mapping.')

mean_profiles = (
    replicate_profiles
    .groupby(['Reference Variable', 'Target Variable', 'Hour'],
             as_index=False, sort=False)['Background-subtracted intensity (ncps)']
    .mean()
)

profile_minima = mean_profiles.groupby('Target Variable')[
    'Background-subtracted intensity (ncps)'
].transform('min')
profile_maxima = mean_profiles.groupby('Target Variable')[
    'Background-subtracted intensity (ncps)'
].transform('max')
profile_ranges = profile_maxima - profile_minima
if (profile_ranges <= 0).any() or profile_ranges.isna().any():
    constant_profiles = mean_profiles.loc[
        profile_ranges <= 0, 'Target Variable'
    ].drop_duplicates().tolist()
    raise ValueError(f'Cannot rescale constant profiles: {constant_profiles}')

mean_profiles['Scaled profile'] = (
    mean_profiles['Background-subtracted intensity (ncps)'] - profile_minima
) / profile_ranges

assert mean_profiles['Target Variable'].nunique() == 38
assert mean_profiles['Reference Variable'].nunique() == 4
assert len(mean_profiles) == 38 * 12

scaling_check = mean_profiles.groupby('Target Variable')['Scaled profile'].agg(['min', 'max'])
assert np.allclose(scaling_check['min'].to_numpy(), 0.0)
assert np.allclose(scaling_check['max'].to_numpy(), 1.0)

print(f'Reconstructed and scaled {mean_profiles["Target Variable"].nunique()} profiles.')
print(f'Profile table: {len(mean_profiles)} rows = 38 targets × 12 oxidation times.')


## Plot the four anchor groups

One graph is generated for each anchor. All profiles within that anchor group are superposed. The anchor is drawn first as a black line with width 4.0; its correlated features are drawn as coloured lines with width 1.2. All graphs share the same time range and scaled 0–1 response range. The four JPG files are the only exported outputs.


In [ ]:
def format_variable_label(variable_name, anchor=False):
    mz_value, mode = parse_variable_name(variable_name)
    label = f'm/z {mz_value:.3f} ({mode})'
    return f'{label} — anchor' if anchor else label


figure_paths = []
expected_group_counts = group_counts.to_dict()

for anchor in ANCHOR_ORDER:
    mapped_targets = target_mapping.loc[
        target_mapping['Reference Variable'] == anchor, 'Target Variable'
    ].tolist()
    correlated_targets = [target for target in mapped_targets if target != anchor]

    fig, ax = plt.subplots(figsize=(11, 7.5))

    anchor_data = mean_profiles[mean_profiles['Target Variable'] == anchor]
    ax.plot(
        anchor_data['Hour'],
        anchor_data['Scaled profile'],
        color='black',
        linewidth=ANCHOR_LINE_WIDTH,
        marker='o',
        markersize=6,
        label=format_variable_label(anchor, anchor=True),
        zorder=10,
    )

    colours = sns.color_palette('tab20', n_colors=len(correlated_targets))
    for colour, target in zip(colours, correlated_targets):
        target_data = mean_profiles[mean_profiles['Target Variable'] == target]
        ax.plot(
            target_data['Hour'],
            target_data['Scaled profile'],
            color=colour,
            linewidth=CORRELATED_LINE_WIDTH,
            alpha=0.85,
            label=format_variable_label(target),
            zorder=3,
        )

    anchor_mz, anchor_mode = parse_variable_name(anchor)
    ax.set_title(
        f'Profiles associated with m/z {anchor_mz:.3f} ({anchor_mode})',
        fontsize=15,
    )
    ax.set_xlabel('Time (Hours)', fontsize=13)
    ax.set_ylabel('Scaled profile (0–1)', fontsize=13)
    ax.set_ylim(-0.03, 1.03)
    ax.set_xticks(sorted(mean_profiles['Hour'].unique()))
    ax.tick_params(axis='x', labelrotation=45)
    ax.grid(True, alpha=0.3)
    ax.legend(
        title='Mass-spectral feature',
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        borderaxespad=0,
        fontsize=8,
        title_fontsize=9,
    )

    fig.tight_layout()
    safe_anchor = anchor.replace('MZ_', 'MZ_').replace('.', 'p')
    figure_path = OUTPUT_ROOT / f'Grouped_profiles_{safe_anchor}.jpg'
    fig.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    if not figure_path.exists() or figure_path.stat().st_size == 0:
        raise RuntimeError(f'Figure was not created correctly: {figure_path}')
    figure_paths.append(figure_path)

assert len(figure_paths) == 4
for anchor, expected_count in expected_group_counts.items():
    observed_count = mean_profiles.loc[
        mean_profiles['Reference Variable'] == anchor, 'Target Variable'
    ].nunique()
    assert observed_count == expected_count

print('Saved figures:')
for figure_path in figure_paths:
    print(f'  {figure_path.resolve()} ({figure_path.stat().st_size:,} bytes)')
